# 05 — Event Study (CAR Analysis)

For each jump definition, compute cumulative abnormal returns (CAR) from T=-60 to T=+120
around every detected jump event. Aggregate by signal direction (+1 / -1).

**Decision gate**: If mean_CAR < 0.3% net of costs at the best horizon for all definitions → strategy not viable.

In [ ]:
import sys
sys.path.append('..')

import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from src.event_study import full_event_study, subsample_analysis
from src.plots import plot_car

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

raw_dir  = Path('../data/raw')
proc_dir = Path('../data/processed')

# Load BTC returns
btc = pd.read_parquet(raw_dir / 'crypto' / 'crypto_1m_BTCUSDT.parquet')
btc = btc.set_index('timestamp_utc').sort_index()
btc_ret = btc['log_ret']

DEFINITIONS = ['D1', 'D2', 'D3', 'D4']
signals_dict = {}
for defn in DEFINITIONS:
    path = proc_dir / f'jump_signals_{defn}.parquet'
    if path.exists():
        signals_dict[defn] = pd.read_parquet(path)

print('Loaded signals:', list(signals_dict.keys()))

## Run Event Study for Each Definition

Pooling all markets together gives the most events (more statistical power).
Then we'll also run per-market to see which markets drive the result.

In [ ]:
event_study_results = {}  # {defn: agg_car_df}

for defn, sig_df in signals_dict.items():
    # Pool all markets: stack all signals into one combined Series
    # Only one market fires at any given time (they're independent), so this is valid
    combined_signal = sig_df.stack()  # MultiIndex (timestamp, market)
    combined_signal = combined_signal[combined_signal != 0]  # only jump events

    # Flatten back to timestamp-indexed Series (take first firing market per minute)
    # If multiple markets fire at the same minute, take the largest magnitude
    if not combined_signal.empty:
        flat_signal = combined_signal.groupby(level=0).apply(
            lambda x: x.iloc[x.abs().argmax()]
        )
        flat_signal = flat_signal.reindex(btc_ret.index).fillna(0)
    else:
        flat_signal = pd.Series(0.0, index=btc_ret.index)

    n_events = int((flat_signal != 0).sum())
    print(f'{defn}: {n_events} pooled events')

    if n_events < 10:
        print(f'  Too few events for event study, skipping {defn}')
        continue

    agg_car, _ = full_event_study(
        signal=flat_signal,
        log_ret=btc_ret,
        pre=60,
        post=120,
        normalize=True,
        vol_window=30,
    )

    event_study_results[defn] = agg_car

print('\nEvent studies complete.')

## Plot CAR for Each Definition

In [ ]:
for defn, agg_car in event_study_results.items():
    if agg_car.empty:
        continue
    fig = plot_car(agg_car, title=f'CAR Around Jump Events — {defn}')
    plt.show()
    plt.close()

## Peak CAR Summary — Decision Gate

In [ ]:
COST_PER_EVENT = 0.001  # 0.1% round trip cost (commission + slippage)
MIN_NET_CAR = 0.003     # 0.3% minimum

print('=== Peak CAR by Definition (Positive Jumps) ===')
viable_definitions = []

for defn, agg_car in event_study_results.items():
    if agg_car.empty:
        continue

    pos = agg_car[agg_car.get('direction', 'positive') == 'positive'] if 'direction' in agg_car.columns else agg_car
    post_event = pos[pos['t'] > 0]

    if post_event.empty:
        continue

    peak_idx = post_event['mean_car'].abs().idxmax()
    peak = post_event.loc[peak_idx]
    net_car = peak['mean_car'] - COST_PER_EVENT

    is_significant = peak['p_value'] < 0.05 if 'p_value' in peak.index else False
    is_viable = net_car >= MIN_NET_CAR and is_significant

    if is_viable:
        viable_definitions.append(defn)

    print(f'  {defn}: peak CAR={peak["mean_car"]*100:.3f}% at T={peak["t"]}min, '
          f'net={net_car*100:.3f}%, p={peak.get("p_value", float("nan")):.3f}, '
          f'viable={is_viable}')

print(f'\nViable definitions to proceed with: {viable_definitions}')
if not viable_definitions:
    print('STOP: No definition shows sufficient net CAR. Strategy not viable.')
else:
    print(f'PROCEED to backtesting with: {viable_definitions}')

## Subsample Analysis

In [ ]:
# Run subsample analysis for the best definition
if viable_definitions:
    best_defn = viable_definitions[0]
    sig_df = signals_dict[best_defn]

    # Flatten signals (same as above)
    combined_signal = sig_df.stack()
    combined_signal = combined_signal[combined_signal != 0]
    flat_signal = combined_signal.groupby(level=0).apply(lambda x: x.iloc[x.abs().argmax()])
    flat_signal = flat_signal.reindex(btc_ret.index).fillna(0)

    subsamples = subsample_analysis(
        signal=flat_signal,
        log_ret=btc_ret,
        btc_ret=btc_ret,
    )

    for label, agg in subsamples.items():
        if agg.empty:
            continue
        print(f'\n--- Subsample: {label} ---')
        post = agg[agg['t'] > 0]
        if not post.empty:
            print(post[['t', 'mean_car', 'p_value', 'n_events']].to_string(index=False))